# Canonical Correlation Analysis of 16S V1-V3 and metabolomics table with top VIPs

Date created: 1/27/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen, Tyler Myers

In [ ]:
import biom
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [776]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [777]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [778]:
# Read in metabolomics table with top VIPs
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)

In [779]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
metaB_tbl = metaB_tbl.drop(columns=['group'])
metaB_tbl

,Urocanic acid,Nicotine,Phenylalanine,D-TRYPTOPHAN,Glutamyltyrosine,C19H36O4 Fatty acid,C19H38O4 Fatty acid,C19H22O4 Linear diarylheptanoids,N-Oleoylethanolamine,Gln-C14:0,Sorbitane Monooleate
SampleID,,,,,,,,,,,
LAMI.RD001.D0.C1,39859.7580,0.00,970487.500,567510.060,12406.8260,611709.700,25936.0230,0.0000,16778.8380,0.0000,0.00
LAMI.RD306.D9.C2,0.0000,178911.47,1071868.200,672054.400,20433.4900,437938.200,23435.4340,6174.2900,12865.3900,0.0000,0.00
LAMI.RD304.D11.C1,36750.4000,0.00,595568.940,328151.120,9403.6520,78491.530,1793.8712,15929.3220,0.0000,5375.8525,99470.69
LAMI.RD304.D11.C2,0.0000,0.00,303845.300,152392.720,2286.2397,261996.800,8281.6990,28877.0040,15780.3090,2691.0256,157503.92
LAMI.RD304.D0.C1,3345.8184,0.00,293386.340,137201.390,0.0000,427202.060,23404.4920,36777.8320,3228.1143,3431.0034,0.00
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,0.00,579058.800,498893.780,5144.6616,122958.260,9814.8600,70822.4450,56654.0430,0.0000,0.00
LAMI.RD308.D9.C3,3076.9688,0.00,281896.700,262886.470,0.0000,65659.700,5371.9844,5548.9746,21826.1170,0.0000,0.00
LAMI.RD319.D2.C2,0.0000,0.00,255542.720,121803.890,0.0000,73231.560,1726.0502,17378.8120,19741.2460,0.0000,0.00


In [780]:
# Ensure the two datasets have the same samples (rows)
common_samples = V1V3_tbl.index.intersection(metaB_tbl.index)
microbiome = V1V3_tbl.loc[common_samples]
metabolomics = metaB_tbl.loc[common_samples]

# Step 1: Preprocess the data
# Apply CLR transformation to each dataset
microbiome_clr = clr(microbiome + 1e-6)  # Add a pseudocount to avoid log of zero
metabolomics_clr = clr(metabolomics + 1e-6)

X = np.array(microbiome_clr)
Y = np.array(metabolomics_clr)

# Step 2: Perform Canonical Correlation Analysis
# Set n_components to the number of canonical dimensions to explore
n_components = min(X.shape[1], Y.shape[1])
cca = CCA(n_components=n_components)
cca.fit(X, Y)

# Get canonical variates
X_c, Y_c = cca.transform(X, Y)

# Step 3: Evaluate the results
canonical_corrs = np.corrcoef(X_c.T, Y_c.T).diagonal(offset=n_components)

# Print the canonical correlations
print("Canonical correlations:", canonical_corrs)

Canonical correlations: [0.8871991  0.73250832 0.72364222 0.66875978 0.64657819 0.64114535
 0.60792486 0.56139161 0.49066077 0.45112947        nan]


/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/sklearn/cross_decomposition/_pls.py:305: UserWarning: Y residual is constant at iteration 10
  warnings.warn(f"Y residual is constant at iteration {k}")
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/yangchen/miniforge3/envs/qiime2-tiny-2024.10+gemelli/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [781]:
# Calculate the correlation coefficient (Pearson's r)
correlation_coefficient = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]

# Step 4: Visualize the results with annotation
plt.figure(figsize=(6, 6))

# Scatter plot for the canonical variates
plt.scatter(X_c[:, 0], Y_c[:, 0], c='purple', label='16S-MetaB matched samples', alpha=0.7, edgecolors='black', linewidth=0.25)

# Add a diagonal line to indicate perfect correlation
plt.plot(plt.gca().get_xlim(), plt.gca().get_ylim(), ls="--", c="gray", linewidth=2)

# Annotate the correlation coefficient on the plot
plt.text(0.04, 0.93, '---', fontsize=22, color='grey', transform=plt.gca().transAxes)
plt.text(0.125, 0.935, f'r = {correlation_coefficient:.2f}', color='black', transform=plt.gca().transAxes, fontsize=14)

# Define tick increments
x_ticks = np.arange(-1, 2.5, 0.5)  # From -1 to 2, inclusive, in steps of 0.5
y_ticks = np.arange(-1, 1.5, 0.5)  # From -1 to 1, inclusive, in steps of 0.5

# Set ticks and labels
plt.xticks(x_ticks, fontsize=12)
plt.yticks(y_ticks, fontsize=12)

# Set axis limits
plt.xlim(-1, 2)
plt.ylim(-1, 1)

# Formatting the plot
plt.axhline(0, color='black', linewidth=0.5, linestyle='--')  # Horizontal axis
plt.axvline(0, color='black', linewidth=0.5, linestyle='--')  # Vertical axis
plt.xlabel('16S Canonical Variates', fontsize=16, labelpad=10)
plt.ylabel('MetaB Canonical Variates', fontsize=16)
plt.title('Multi-omics Canonical Correlation', fontsize=20)
plt.legend(loc='upper left', bbox_to_anchor=(0.42, 0.1))
plt.tight_layout()

# Save the figure
plt.savefig('../Figures/multi-omics_Figures/biplot_with_correlation.png', dpi=600)

In [782]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl = V1V3_tbl[V1V3_tbl.index.isin(metaB_tbl.index)]
top_V1V3_features = [' g__Cutibacterium', ' g__uncultured', ' g__Staphylococcus',
                     ' g__Streptococcus', ' g__Corynebacterium', ' g__Lawsonella',
                     ' g__Micrococcus', ' g__Alloprevotella', ' g__Lactobacillus', ' g__Rothia']
# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl.columns.intersection(top_V1V3_features)
V1V3_tbl = V1V3_tbl[available_columns]
V1V3_tbl

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,g__Rothia
LAMI.RD001.D0.C1,2362.0,0.0,94.0,292.0,175.0,91.0,37.0,18.0,1.0,50.0
LAMI.RD001.D0.C2,1808.0,2.0,158.0,374.0,352.0,120.0,52.0,27.0,165.0,43.0
LAMI.RD001.D14.C1,2234.0,2.0,83.0,240.0,492.0,253.0,30.0,30.0,60.0,12.0
LAMI.RD001.D14.C2,1761.0,0.0,137.0,446.0,456.0,151.0,16.0,100.0,26.0,31.0
LAMI.RD001.D28.C1,2367.0,11.0,118.0,293.0,365.0,217.0,22.0,38.0,17.0,14.0
...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,981.0,2759.0,21.0,0.0,2.0,5.0,0.0,0.0,0.0,0.0
LAMI.RD318.D28.C3,3218.0,538.0,5.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1900.0,1846.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,1410.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0


In [783]:
# # Read in metabolomics metadata
# VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_name.csv')

# # Create a mapping from the columns in metaB_tbl to the 'class' column in VIP_filtered
# feature_to_class = dict(zip(VIP_filtered['Shortened_Compound_Name'], VIP_filtered['class']))

# # Replace column names in metaB_tbl using the mapping
# metaB_tbl = metaB_tbl.rename(columns=feature_to_class)

# # Average columns with the same name
# metaB_tbl = metaB_tbl.groupby(metaB_tbl.columns, axis=1).mean()

# metaB_tbl

In [784]:
# Read in metabolomics table
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/data_sample_10282024.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Subset to matching samples
metaB_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Cast column names to integers
metaB_tbl.columns = metaB_tbl.columns.astype(int)

metaB_tbl

,4293,15974,2531,5907,5946,2965,22668,1032,1074,955,...,24241,25841,24173,30835,24180,25998,33060,30635,26562,28336
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,4770.76270,0.0000,72812.695,1022406.25,0.0,383280.300,35353.773,44675.1600,9472625.0,55584.406,...,0.00,0.000,0.00,0.00,0.000,98869.660,0.0000,0.0000,44668.117,8376.1850
LAMI.RD306.D9.C2,10916.94300,0.0000,156774.600,340379.90,0.0,412205.800,55508.960,22453.7930,11737546.0,56046.720,...,79494.93,0.000,62701.67,0.00,55628.926,40144.098,0.0000,0.0000,42782.950,11809.1740
LAMI.RD304.D11.C1,2342.99900,0.0000,116903.530,548906.90,0.0,232978.390,36155.680,47782.8440,11899350.0,51932.010,...,0.00,84496.870,0.00,0.00,0.000,106990.060,0.0000,0.0000,66076.950,8903.7910
LAMI.RD304.D11.C2,740.01514,6008.3945,69846.520,303948.28,0.0,114657.980,52892.070,24622.6290,5926156.5,25971.598,...,0.00,25730.426,0.00,0.00,0.000,55033.453,1521.3207,0.0000,19372.977,8350.3290
LAMI.RD304.D0.C1,725.83435,6228.5117,75098.500,438357.53,0.0,106253.080,45601.550,6258.2560,3217912.8,18388.262,...,0.00,28067.354,0.00,0.00,0.000,107743.010,0.0000,0.0000,49133.240,10825.2950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.00000,0.0000,53799.130,458503.40,0.0,219629.830,22518.434,12877.7910,3450308.5,16107.057,...,0.00,0.000,0.00,0.00,0.000,40761.030,0.0000,0.0000,15749.498,5134.8340
LAMI.RD308.D9.C3,0.00000,0.0000,30274.117,1192993.40,0.0,98506.914,0.000,0.0000,398077.2,0.000,...,0.00,0.000,0.00,39292.41,0.000,14812.991,0.0000,0.0000,0.000,0.0000
LAMI.RD319.D2.C2,1567.89330,0.0000,63744.797,458884.62,0.0,92721.800,25671.117,3202.1710,1813529.9,4184.649,...,0.00,0.000,0.00,0.00,0.000,100497.234,0.0000,0.0000,0.000,2682.9302


In [785]:
# Read in VIP filtered table
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_all.csv')
VIP_filtered

,Unnamed: 0,Feature,VIP,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,...,InChIKey,InChIKey-Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group
0,97,5931,2.495720,CCMSLIB00003139168,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.908552,1395700.0,0,1.541540,...,NaN,NaN,Unknown,Unknown,Unknown,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003139168,NL vs L
1,98,5930,2.460781,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900.0,0,0.520785,...,QIVBCDIJIAJPQS-SECBINFHSA-N,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L
2,99,22234,2.294601,CCMSLIB00005884986,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.782720,436400.0,0,2.131170,...,PSHXNVGSVNEJBD-LJQANCHMSA-O,PSHXNVGSVNEJBD,Organic nitrogen compounds,Organonitrogen compounds,Quaternary ammonium salts,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005884986,NL vs L
3,0,29059,2.287412,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000.0,0,1.960920,...,LUIGTZGBXWZJAX-UHFFFAOYSA-N,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L
4,47,29059,2.215677,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000.0,0,1.960920,...,LUIGTZGBXWZJAX-UHFFFAOYSA-N,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,144,15949,0.019873,CCMSLIB00005721447,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.862653,2910700.0,0,0.275607,...,ZBFSUZGUYFFWGY-UHFFFAOYSA-N,ZBFSUZGUYFFWGY,Phenylpropanoids and polyketides,Diarylheptanoids,Linear diarylheptanoids,Diarylheptanoids,Linear diarylheptanoids,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005721447,NL vs L
143,96,655,0.017956,CCMSLIB00006681721,spectra_filtered.mgf,MONA.mgf,0.804832,4820000.0,0,62.549500,...,LOIYMIARKYCTBW-OWOJBTEDSA-N,LOIYMIARKYCTBW,Organoheterocyclic compounds,Azoles,Imidazoles,NaN,NaN,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00006681721,H vs NL
144,145,12004,0.013883,CCMSLIB00012254879,spectra_filtered.mgf,CMMC-REFRAME-POSITIVE-LIBRARY.mgf,0.941819,685800.0,0,10.966100,...,ZAKOWWREFLAJOT-UHFFFAOYSA-N,ZAKOWWREFLAJOT,Lipids and lipid-like molecules,Prenol lipids,Quinone and hydroquinone lipids,Meroterpenoids,Prenyl quinone meroterpenoids,Terpenoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012254879,NL vs L
145,46,14633,0.009475,CCMSLIB00000848791,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.888533,3972000.0,0,6.254110,...,GZVIQGVWSNEONZ-UHFFFAOYSA-N,GZVIQGVWSNEONZ,Phenylpropanoids and polyketides,Diarylheptanoids,Linear diarylheptanoids,Diarylheptanoids,Linear diarylheptanoids,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000848791,H vs L


In [786]:
# Ensure both metaB_tbl columns and VIP_filtered['Feature'] are strings
metaB_tbl.columns = metaB_tbl.columns.astype(str)
VIP_filtered['Feature'] = VIP_filtered['Feature'].astype(str)

# Identify columns to keep (those present in VIP_filtered['Feature'])
columns_to_keep = metaB_tbl.columns.intersection(VIP_filtered['Feature'])

# Subset metaB_tbl to keep only these columns
metaB_tbl = metaB_tbl[columns_to_keep]

# Verify the updated DataFrame
metaB_tbl


,1454,655,5062,777,2969,5931,1821,142,5930,3496,...,28319,24485,19710,24392,7323,11925,12098,11163,13564,471
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,16695.8440,39859.7580,28277.594,0.00,970487.500,436126.500,11768.7170,14594.9800,567510.060,1473068.10,...,39880.5400,15827.123,0.000,0.00,127401.1600,125692.330,222831.060,93022.8000,9556.500,0.000
LAMI.RD306.D9.C2,18734.0000,0.0000,253083.810,178911.47,1071868.200,495163.720,11677.9920,0.0000,672054.400,308019.97,...,1761.0082,0.000,0.000,0.00,79131.6100,10553.927,0.000,8086.6416,9680.064,13462.965
LAMI.RD304.D11.C1,11413.2820,36750.4000,15391.807,0.00,595568.940,253283.060,7820.1587,0.0000,328151.120,299140.06,...,12144.8260,0.000,23546.951,99470.69,7700.7490,12473.092,0.000,0.0000,20212.285,0.000
LAMI.RD304.D11.C2,11557.3930,0.0000,15015.189,0.00,303845.300,121623.650,8872.6210,18229.2730,152392.720,191713.67,...,8737.5550,0.000,34863.656,157503.92,12418.2560,40673.110,295772.030,33420.1170,13173.599,0.000
LAMI.RD304.D0.C1,0.0000,3345.8184,45187.030,0.00,293386.340,105660.820,10458.3950,16433.0470,137201.390,705570.80,...,2826.6750,0.000,0.000,0.00,8974.0370,49464.902,21532.973,6135.0205,22111.420,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,12858.4660,9787.1210,14712.559,0.00,579058.800,370028.560,9088.1910,20261.2170,498893.780,236111.75,...,71891.6250,0.000,0.000,0.00,0.0000,0.000,0.000,0.0000,0.000,0.000
LAMI.RD308.D9.C3,14476.2890,3076.9688,17934.553,0.00,281896.700,189676.940,26168.9120,6373.1865,262886.470,303509.40,...,2425.2832,0.000,0.000,0.00,6148.7803,220846.160,12894.838,0.0000,30434.875,0.000
LAMI.RD319.D2.C2,14740.0510,0.0000,10556.326,0.00,255542.720,96321.195,19116.4140,0.0000,121803.890,5406879.00,...,78557.1700,0.000,0.000,0.00,32402.7340,25695.014,92320.030,101395.3900,0.000,23633.705


In [787]:
# Get the intersection of columns in metaB_tbl and Feature in VIP_filtered
matching_features = set(metaB_tbl.columns).intersection(set(VIP_filtered['Feature']))
print(f"Number of matching features: {len(matching_features)}")
print(f"Matching features: {matching_features}")

# Ensure both are strings
metaB_tbl.columns = metaB_tbl.columns.astype(str)
VIP_filtered['Feature'] = VIP_filtered['Feature'].astype(str)

# Strip leading/trailing whitespace from both sides
metaB_tbl.columns = metaB_tbl.columns.str.strip()
VIP_filtered['Feature'] = VIP_filtered['Feature'].str.strip()

# Create the mapping
feature_to_class = dict(zip(VIP_filtered['Feature'], VIP_filtered['class']))

# Rename columns in metaB_tbl
metaB_tbl = metaB_tbl.rename(columns=feature_to_class)

# Avgerage columns with the same name
metaB_tbl = metaB_tbl.groupby(metaB_tbl.columns, axis=1).mean()

# Verify the result
metaB_tbl



Number of matching features: 50
Matching features: {'853', '18476', '15963', '5930', '11814', '15770', '7323', '7778', '3496', '29059', '24392', '15949', '1821', '14657', '19710', '4546', '655', '13564', '11163', '24485', '322', '15947', '5062', '471', '26788', '2969', '1454', '28600', '12004', '27758', '20989', '12098', '11925', '18483', '22718', '15022', '2617', '142', '30129', '31272', '5931', '777', '14633', '3794', '23297', '28319', '25614', '22234', '5873', '21138'}


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_65799/3597743251.py:21: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metaB_tbl = metaB_tbl.groupby(metaB_tbl.columns, axis=1).mean()


,Azoles,Benzene and substituted derivatives,Benzimidazoles,Carboxylic acids and derivatives,Diarylheptanoids,Diazines,Fatty Acyls,Flavonoids,Imidazopyrimidines,Indoles and derivatives,"Linear 1,3-diarylpropanoids",Organonitrogen compounds,Organooxygen compounds,Phenols,Prenol lipids,Pyridines and derivatives,Unknown
SampleID,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,39859.7580,21376.83000,0.000,619902.43320,8495.205000,7502.30350,316556.542000,0.000,16695.8440,567510.060,2762.87000,8389.41900,42467.053333,28277.594,42919.799333,0.00,137493.755222
LAMI.RD306.D9.C2,0.0000,32936.38964,0.000,413514.68600,103697.917500,0.00000,98388.624250,0.000,18734.0000,672054.400,64806.41850,6432.69500,30864.858333,253083.810,143669.607200,178911.47,130982.925889
LAMI.RD304.D11.C1,36750.4000,10380.92320,0.000,781054.04890,12472.100500,7018.70115,77419.106650,23546.951,11413.2820,328151.120,0.00000,5635.44700,2566.916333,15391.807,9214.140000,0.00,69747.405533
LAMI.RD304.D11.C2,0.0000,14995.59010,0.000,595691.21306,35800.233875,4976.36050,140698.219875,34863.656,11557.3930,152392.720,0.00000,9040.41340,4139.418667,15015.189,17548.005667,0.00,40075.225867
LAMI.RD304.D0.C1,3345.8184,3721.89100,0.000,219740.44468,26135.974250,0.00000,153396.903375,0.000,0.0000,137201.390,0.00000,1614.05715,2991.345667,45187.030,8859.422167,0.00,39982.620111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,14378.32500,0.000,448481.11232,33775.238750,0.00000,46734.046925,0.000,12858.4660,498893.780,1887.60200,28327.02150,0.000000,14712.559,3029.468000,0.00,134235.268256
LAMI.RD308.D9.C3,3076.9688,485.05664,658741.100,147938.47800,25311.123650,24080.56400,76759.694800,0.000,14476.2890,262886.470,56814.02800,23193.68550,4690.023300,17934.553,3688.485000,0.00,58211.979056
LAMI.RD319.D2.C2,0.0000,15711.43400,9720.738,283929.56200,4344.703000,22323.59200,699981.456775,0.000,14740.0510,121803.890,757.83075,13242.45750,18678.813000,10556.326,41264.133667,0.00,33195.782311


In [788]:
# Subset to only Acne_L samples
group_name = 'Healthy'
samples_keep = metadata.loc[metadata['group'] == group_name, 'SampleID']
metaB_tbl = metaB_tbl.loc[metaB_tbl.index.intersection(samples_keep)]

V1V3_tbl = V1V3_tbl.loc[V1V3_tbl.index.intersection(samples_keep)]

In [789]:
# Ensure the two datasets have the same samples (rows)
common_samples = V1V3_tbl.index.intersection(metaB_tbl.index)
microbiome = V1V3_tbl.loc[common_samples]
metabolomics = metaB_tbl.loc[common_samples]

# Step 1: Preprocess the data
# Apply CLR transformation to each dataset
microbiome_clr = clr(microbiome + 1)  # Add a pseudocount to avoid log of zero
metabolomics_clr = clr(metabolomics + 1)

X = np.array(microbiome_clr)
Y = np.array(metabolomics_clr)

# Step 2: Perform Canonical Correlation Analysis
# Set n_components to the number of canonical dimensions to explore
n_components = min(X.shape[1], Y.shape[1])
cca = CCA(n_components=n_components)
cca.fit(X, Y)

# Get canonical variates
X_c, Y_c = cca.transform(X, Y)

# Step 3: Evaluate the results
canonical_corrs = np.corrcoef(X_c.T, Y_c.T).diagonal(offset=n_components)

# Print the canonical correlations
print("Canonical correlations:", canonical_corrs)

Canonical correlations: [0.93941722 0.8685767  0.78579031 0.77872996 0.7432519  0.63535245
 0.60632136 0.45093068 0.25316489 0.03918953]


In [790]:
# Evaluate canonical loadings
microbiome_loadings = cca.x_weights_
metabolomics_loadings = cca.y_weights_

# Rank features by contribution
top_microbiome_features = np.argsort(np.abs(microbiome_loadings[:, 0]))[::-1]
top_metabolite_features = np.argsort(np.abs(metabolomics_loadings[:, 0]))[::-1]

# Extract feature names
microbiome_features = microbiome.columns[top_microbiome_features]
metabolite_features = metabolomics.columns[top_metabolite_features]

In [791]:
# Compute pairwise correlations between 16S and metabolites
correlation_matrix = np.corrcoef(X.T, Y.T)[:X.shape[1], X.shape[1]:]

# Plot the heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(
    correlation_matrix, 
    cmap='coolwarm', 
    annot=False, 
    fmt=".2f", 
    xticklabels=metabolomics.columns, 
    yticklabels=microbiome.columns, 
    cbar_kws={'label': 'Correlation Coefficient'}
)

# Format the plot
plt.title(f'Top Correlations Between 16S Features and Metabolite Classes for {group_name}', fontsize=16)
plt.xlabel('Metabolite Features', fontsize=14)
plt.ylabel('16S Features', fontsize=14)
plt.xticks(rotation=90, fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()

# Save the figure
plt.savefig(f'../Figures/multi-omics_Figures/top_correlations_heatmap_{group_name}.png', dpi=600)
